In [11]:
from google.colab import drive
import os, torch, timm, numpy as np, zipfile, random, shutil
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/DriveGuard'
WEIGHTS_FILENAME = 'models/vit_spatial_model_v1.pth'
DATASET_ZIP_FILENAME = 'all_cams_ds_driveguard_temporal_roi.zip'

LOCAL_WEIGHTS = '/content/vit_spatial_model_v1.pth'
LOCAL_ZIP = '/content/dataset_temporal.zip'
DATASET_PATH = '/content/ds_driveguard_temporal_roi'
OUTPUT_FEATURES_DIR = '/content/drive/MyDrive/DriveGuard/ViT_Features/'

device = torch.device("cuda")

def prepare_files():
    src_weights = os.path.join(DRIVE_PATH, WEIGHTS_FILENAME)
    src_zip = os.path.join(DRIVE_PATH, DATASET_ZIP_FILENAME)

    if os.path.exists(src_weights) and not os.path.exists(LOCAL_WEIGHTS):
        shutil.copy(src_weights, LOCAL_WEIGHTS)

    if os.path.exists(src_zip) and not os.path.exists(LOCAL_ZIP):
        shutil.copy(src_zip, LOCAL_ZIP)

prepare_files()

if os.path.exists(LOCAL_ZIP) and not os.path.exists(DATASET_PATH):
    with zipfile.ZipFile(LOCAL_ZIP, 'r') as zip_ref:
        zip_ref.extractall('/content/')

os.makedirs(OUTPUT_FEATURES_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
time: 1.34 s (started: 2026-04-04 18:41:22 +00:00)


In [12]:
!pip install ipython-autotime
%load_ext autotime

Exception in thread QueueFeederThread:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 259, in _feed
Exception ignored in: <function _ConnectionBase.__del__ at 0x79f3c11c1440>
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 133, in __del__
Exception in thread QueueFeederThread:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 259, in _feed
    reader_close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    reader_close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor
    self._close()
  File "/usr/lib/python3.12/multiproc

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 4.35 s (started: 2026-04-04 18:41:26 +00:00)


In [13]:
# Cell 2
class SequenceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        for split in ['train', 'val', 'test']:
            split_path = os.path.join(root_dir, split)
            if not os.path.exists(split_path): continue
            for cls in ['Drink', 'Phone', 'Safe']:
                cls_path = os.path.join(split_path, cls)
                if not os.path.isdir(cls_path): continue
                for clip in os.listdir(cls_path):
                    clip_full_path = os.path.join(cls_path, clip)
                    if os.path.isdir(clip_full_path):
                        self.samples.append({
                            'path': clip_full_path, 'split': split,
                            'class': cls, 'name': clip
                        })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        frames = sorted([f for f in os.listdir(s['path']) if f.endswith('.jpg')])[:16]
        imgs = [self.transform(Image.open(os.path.join(s['path'], f)).convert('RGB')) for f in frames]
        return torch.stack(imgs), s['split'], s['class'], s['name']

preprocess = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

time: 2.1 ms (started: 2026-04-04 18:43:31 +00:00)


In [14]:
# Cell 3
def get_optimized_extractor(path):
    model = timm.create_model('vit_so400m_patch14_siglip_384', pretrained=False, num_classes=3)
    model.load_state_dict(torch.load(path, map_location=device))
    model.reset_classifier(0)
    model = model.to(device).to(torch.bfloat16).eval()
    try:
        model = torch.compile(model)
    except:
        pass
    return model

extractor = get_optimized_extractor(LOCAL_WEIGHTS)

time: 7.08 s (started: 2026-04-04 18:43:34 +00:00)


In [ ]:
# Cell 4
def run_extraction(batch_size=4):
    dataset = SequenceDataset(DATASET_PATH, transform=preprocess)
    loader = DataLoader(dataset, batch_size=batch_size, num_workers=4, pin_memory=True)

    with torch.no_grad():
        for seq_batch, splits, classes, names in tqdm(loader):
            B, T, C, H, W = seq_batch.shape
            flat_batch = seq_batch.view(-1, C, H, W).to(device, dtype=torch.bfloat16)

            with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
                features = extractor(flat_batch)

            features = features.view(B, T, -1).cpu().float().numpy()

            for i in range(B):
                out_dir = os.path.join(OUTPUT_FEATURES_DIR, splits[i], classes[i])
                if not os.path.exists(out_dir):
                    os.makedirs(out_dir, exist_ok=True)
                save_path = os.path.join(out_dir, f"{names[i]}.npy")
                if not os.path.exists(save_path):
                    np.save(save_path, features[i])

run_extraction(batch_size=4)

  0%|          | 0/4255 [00:00<?, ?it/s]

W0404 18:44:04.954000 19603 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


In [10]:
# Cell 5
def final_verification(base_path):
    all_files = []
    for root, _, files in os.walk(base_path):
        for file in files:
            if file.endswith(".npy"):
                all_files.append(os.path.join(root, file))

    if not all_files:
        return

    test_file = random.choice(all_files)
    data = np.load(test_file)

    print(f"File: {os.path.basename(test_file)}")
    print(f"Shape: {data.shape}")
    print(f"Valid: {not np.isnan(data).any()}")

    plt.figure(figsize=(10, 4))
    plt.hist(data.flatten(), bins=100, color='purple')
    plt.show()

final_verification(OUTPUT_FEATURES_DIR)

time: 2.33 ms (started: 2026-04-04 18:21:15 +00:00)
